In [6]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# LOAD DATASET

df = pd.read_csv(r"C:\Users\Admin\Desktop\PROJECT FILE\cybersecurity_intrusion_data.csv") 

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# PREPARE DATA

target_column = "login_attempts"

# Find numeric columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

# Keep target column
if target_column not in numeric_cols:
    numeric_cols.append(target_column)

df = df[numeric_cols]

print("\nRemaining Columns:")
print(df.columns.tolist())

X = df.drop(columns=[target_column])
y = df[target_column]
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# TRAIN MODEL

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

# EVALUATE MODEL

pred = model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, pred))

# SAVE MODEL

joblib.dump(model, "threat_detection_model.pkl")

print("\nModel saved as threat_detection_model.pkl")

# THREAT TYPES

THREAT_TYPES = {
    0: "Normal",
    1: "DoS",
    2: "Probe",
    3: "R2L",
    4: "U2R",
    5: "Malware"
}

# THREAT DETECTION FUNCTION

def classify_threat(model, input_data):
    prediction = model.predict(input_data)[0]

    if prediction in THREAT_TYPES:
        return THREAT_TYPES[prediction]

    return prediction

# TEST THREAT DETECTION

sample = X_test.iloc[[0]]

threat = classify_threat(model, sample)

print("\nDetected Threat:", threat)

# RISK SCORE + SEVERITY

probabilities = model.predict_proba(sample)[0]

confidence = probabilities.max()

risk_score = round(confidence * 100, 2)

if risk_score < 30:
    severity = "Low"
elif risk_score < 70:
    severity = "Medium"
else:
    severity = "High"

print("Risk Score:", risk_score)
print("Severity:", severity)

result = {
    "threat_type": threat,
    "risk_score": risk_score,
    "severity": severity
}

print("\nResult:")
print(result)

Dataset Shape: (9537, 11)

Columns:
['session_id', 'network_packet_size', 'protocol_type', 'login_attempts', 'session_duration', 'encryption_used', 'ip_reputation_score', 'failed_logins', 'browser_type', 'unusual_time_access', 'attack_detected']

Remaining Columns:
['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access', 'attack_detected']

Classification Report:
              precision    recall  f1-score   support

           1       0.13      0.07      0.09       183
           2       0.14      0.12      0.13       272
           3       0.19      0.23      0.21       366
           4       0.23      0.29      0.26       370
           5       0.17      0.19      0.18       302
           6       0.09      0.06      0.07       197
           7       0.21      0.28      0.24       119
           8       0.09      0.05      0.07        55
           9       0.06      0.05      0.05        22
          10       0.00  

C:\Users\Admin\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Model saved as threat_detection_model.pkl

Detected Threat: U2R
Risk Score: 60.0
Severity: Medium

Result:
{'threat_type': 'U2R', 'risk_score': 60.0, 'severity': 'Medium'}


In [9]:
import pandas as pd
import numpy as np
import joblib

# Load trained model
model = joblib.load("threat_detection_model.pkl")

# Load data
data = pd.read_csv(
    r"C:\Users\Admin\Desktop\PROJECT FILE\cybersecurity_intrusion_data.csv"
)

# Target column used during training
target_column = "login_attempts"

# Keep only numeric columns
numeric_cols = data.select_dtypes(include=['number']).columns.tolist()

# Remove target column if it exists
if target_column in numeric_cols:
    numeric_cols.remove(target_column)

# Features for prediction
X = data[numeric_cols]

print("Features used:")
print(X.columns.tolist())

# Make predictions
predictions = model.predict(X)

# Get prediction probabilities
probabilities = model.predict_proba(X)

# Analyze each record
for i in range(len(X)):

    threat_type = predictions[i]

    confidence = np.max(probabilities[i])

    risk_score = round(confidence * 100, 2)

    if risk_score < 30:
        severity = "Low"
    elif risk_score < 70:
        severity = "Medium"
    else:
        severity = "High"

    print(f"\n{'='*40}")
    print(f"Record {i+1}")
    print(f"Threat Type: {threat_type}")
    print(f"Risk Score: {risk_score}%")
    print(f"Confidence Level: {confidence:.4f}")
    print(f"Severity: {severity}")

Features used:
['network_packet_size', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access', 'attack_detected']

Record 1
Threat Type: 3
Risk Score: 30.5%
Confidence Level: 0.3050
Severity: Medium

Record 2
Threat Type: 3
Risk Score: 81.0%
Confidence Level: 0.8100
Severity: High

Record 3
Threat Type: 3
Risk Score: 64.0%
Confidence Level: 0.6400
Severity: Medium

Record 4
Threat Type: 2
Risk Score: 18.5%
Confidence Level: 0.1850
Severity: Low

Record 5
Threat Type: 5
Risk Score: 72.0%
Confidence Level: 0.7200
Severity: High

Record 6
Threat Type: 5
Risk Score: 72.5%
Confidence Level: 0.7250
Severity: High

Record 7
Threat Type: 4
Risk Score: 70.0%
Confidence Level: 0.7000
Severity: High

Record 8
Threat Type: 3
Risk Score: 70.5%
Confidence Level: 0.7050
Severity: High

Record 9
Threat Type: 4
Risk Score: 29.0%
Confidence Level: 0.2900
Severity: Low

Record 10
Threat Type: 6
Risk Score: 81.0%
Confidence Level: 0.8100
Severity: High

Record 11
Threat Type: 2


Risk Score: 67.0%
Confidence Level: 0.6700
Severity: Medium

Record 3518
Threat Type: 4
Risk Score: 74.0%
Confidence Level: 0.7400
Severity: High

Record 3519
Threat Type: 6
Risk Score: 61.0%
Confidence Level: 0.6100
Severity: Medium

Record 3520
Threat Type: 3
Risk Score: 33.5%
Confidence Level: 0.3350
Severity: Medium

Record 3521
Threat Type: 3
Risk Score: 27.0%
Confidence Level: 0.2700
Severity: Low

Record 3522
Threat Type: 7
Risk Score: 62.0%
Confidence Level: 0.6200
Severity: Medium

Record 3523
Threat Type: 3
Risk Score: 72.0%
Confidence Level: 0.7200
Severity: High

Record 3524
Threat Type: 4
Risk Score: 77.0%
Confidence Level: 0.7700
Severity: High

Record 3525
Threat Type: 2
Risk Score: 41.0%
Confidence Level: 0.4100
Severity: Medium

Record 3526
Threat Type: 3
Risk Score: 74.5%
Confidence Level: 0.7450
Severity: High

Record 3527
Threat Type: 2
Risk Score: 68.5%
Confidence Level: 0.6850
Severity: Medium

Record 3528
Threat Type: 6
Risk Score: 72.5%
Confidence Level: 0.7250
